# Reproduction, verification and reviewer map

Entry point for rerunning the complete study and connecting reviewer questions to executed evidence.

**Mode:** executed analysis of the committed corrected results. Expensive model refitting is available through Notebook 11 and `scripts/reproduce.py`. Replaying saved results is not presented as fresh model training.

In [1]:
from pathlib import Path
import sys, json, itertools
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.revision-repository').exists())
sys.path.insert(0, str(ROOT / 'analysis'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from notebook_support import *
from analysis import PROTEINS, TARGET, FIXED, global_quantile, local_intervals
pd.set_option('display.max_columns', 14)
pd.set_option('display.max_rows', 30)
pd.set_option('display.precision', 5)


## Replay or recompute

The committed notebook outputs show verified saved results. To train again, run the command below in a fresh environment. It reconstructs identity, creates clusters and partitions, fits the models, recalibrates intervals, recomputes controls/SHAP/permutations and renders figures. Raw external docking/ADMET calculations are outside this pipeline.

```bash
python -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
python scripts/reproduce.py --output reproduction/my-run
```

The output folder must be new. `--resume` is accepted only when the code/data signature matches. Changed code or data must use a new folder. Do not run fresh calculations into committed results/.

In [2]:
RECOMPUTE_ALL = False  # Explicit opt-in: trains the complete corrected experiment suite.
if RECOMPUTE_ALL:
    import subprocess, datetime
    out=ROOT/'reproduction'/datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
    subprocess.run([sys.executable,str(ROOT/'scripts/reproduce.py'),'--output',str(out)],cwd=ROOT,check=True)
else:
    print('Replay mode: no model refitting requested. All saved model predictions were checked in Notebook 03.')

Replay mode: no model refitting requested. All saved model predictions were checked in Notebook 03.


## Recorded verification

Model predictions, per-compound interval statistics, no-overlap checks, marker-selection invariance to held-out labels, fingerprint identity and SHAP additivity are checked separately. Advisory model responses are not evidence.

In [3]:
for name,path in [('Corrected experiments',RESULTS/'verification.json'),('Additional checks',ADDITIONAL/'verification.json')]:
    record=read_json(path)
    summary={k:(f'{len(v)} records; see linked JSON' if isinstance(v,(list,dict)) else v) for k,v in record.items()}
    display(Markdown('### '+name))
    display(pd.Series(summary,name='Verification').to_frame())
p=ROOT/'provenance/portability_validation.json'
if p.exists():
    record=read_json(p)
    display(pd.Series({k:v for k,v in record.items() if k!='checks'},name='Fresh full run').to_frame())

### Corrected experiments

,Verification
confidence_runs,18 records; see linked JSON
global_model_runs_verified,25
n_primary,12584
identity_manifest_checked,True
holdout_label_perturbation_does_not_change_marker_selection,True
quantile_oracles,True
repaired_dataset_sha256,44c1646daaec0ab5037032d435e526d3e532d98c468249...
note,These checks verify recorded computation and c...
strict_global_models_verified,10
repaired_model_fingerprints_match_chiral_Morgan,3 records; see linked JSON


### Additional checks

,Verification
five_seeds_complete,True
raw_kernel_matches_saved_intervals,True
saved_model_predictions_match,True
BBB_test_ids_and_targets_identical,True
all_target_docking_scores_finite,True
normalization_fit,proper training only
no_model_refitting,True


,Fresh full run
source,"Fresh independent output directory, complete p..."
status,complete
compared_artifacts,409
not_regenerated_by_pipeline,[]
comparison_tolerance,"{'rtol': 1e-08, 'atol': 1e-10}"


## Reviewer-to-experiment map

In [4]:
display(pd.read_csv(ROOT/'docs/reviewer_experiment_map.csv'))

,reviewer,point,topic,notebooks,evidence,status
0,1,1,Sample accounting and structural overlap,01;02;10,data identity;split manifests;fixed-test analo...,completed
1,1,2,Heterogeneous labels and coverage uncertainty,04;08,coverage and cluster bootstrap;source metadata...,completed within available metadata
2,1,3,Calibration versus display bandwidth and fallback,04;08;09,fallback sensitivity;mask fractions;separate b...,completed
3,1,4,BTox validity and collinearity,07;09,VIF;correlations;residual deciles;separate cal...,completed
4,1,5,Mean docking score and normalization,07,target-wise normalization;incremental docking ...,completed
5,1,6,BBB baseline on matched test compounds,06,paired Plain/Baseline/specialist predictions,completed
6,1,7,SHAP and modality reliance,05,group permutation importance;SHAP additivity,completed;no matched-hyperparameter retraining...
7,1,8,Reproducibility and threshold stability,02;09;11,protocols;models;splits;bootstrap thresholds;c...,completed;historical cutoff unknown
8,2,1,Introduction objectives and metric explanations,00;03;04,docs/manuscript_edits_en.md,prepared manuscript text
9,2,2,Conceptual uncertainty background,04,docs/manuscript_edits_en.md,prepared manuscript text


## Boundaries of the revision

No new Optuna optimization, independent prospective/external evaluation, laboratory/year stratification, target-specific biological weighting or causal mechanistic validation was performed. Revised manuscript paragraphs and author replies are prepared in docs/. Historical correspondence is not evidence that those edits have been submitted or incorporated into the journal manuscript.

Full reviewer reports and live Google Doc metadata are not needed to reproduce this repository and are not included.